In [1]:
from pathlib import Path
import time

from sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing import InstancePipeline, InstanceParams, PipelineConfig
from sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing.stations.providers import DonkeyRepublicProvider
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)
logging.getLogger("sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing.routing.route_builder").setLevel(logging.INFO)

instance_dir = Path("data/sum_nsm_dynamic_pricing/geneva_1km_mon08h_geometry")

def main() -> None:
    total_start = time.perf_counter()

    config = PipelineConfig(
        center_lat=46.20604780106243,
        center_lon=6.147834100933916,
        radius_m=1000,
        output_dir=instance_dir,
        gtfs_path=Path("/Users/rebecamurillo/Documents/INRIA/development/inocs-sum-gtfs-geojson/otp/data/geneva-2026-gtfs.zip"),
        gbfs_path=None,
        seed=42,
        otp_timestamp="2026-03-08T08:00:00.000000Z",
        otp_workers=10
    )
    instance_params = InstanceParams()

    pipeline = InstancePipeline(config=config, gbfs_provider=DonkeyRepublicProvider())

    start = time.perf_counter()
    stations_path = pipeline.prepare_stations()
    print(f"[TIMING] prepare_stations: {time.perf_counter() - start:.2f}s")

    start = time.perf_counter()
    hubs_path = pipeline.build_hubs(stations_path)
    print(f"[TIMING] build_hubs: {time.perf_counter() - start:.2f}s")

    start = time.perf_counter()
    routes_path = pipeline.compute_routes(hubs_path)
    print(f"[TIMING] compute_routes: {time.perf_counter() - start:.2f}s")
    print(f"Starting route quality validation for: {routes_path}")
    validated = pipeline.quality_routes(routes_path)
    print(f"Completed route quality validation for: {routes_path} - Validated: {validated}")

    start = time.perf_counter()
    instance_path = pipeline.generate_instance(instance_params)
    print(f"[TIMING] generate_instance: {time.perf_counter() - start:.2f}s")

    print(f"Instance generated at: {instance_path}")
    print(f"[TIMING] total: {time.perf_counter() - total_start:.2f}s")


if __name__ == "__main__":
    main()


[TIMING] prepare_stations: 0.41s


2026-03-25 16:55:42,235 [INFO] pyogrio._io: Created 30 records
2026-03-25 16:55:42,240 [INFO] pyogrio._io: Created 327 records
2026-03-25 16:55:42,251 [INFO] pyogrio._io: Created 30 records


[TIMING] build_hubs: 2.89s


2026-03-25 16:55:43,190 [INFO] sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing.routing.route_builder: Route building: 698 total OD pairs | 0 already cached | 698 to query (workers=10, delay=0.05s)
2026-03-25 16:55:49,582 [INFO] sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing.routing.route_builder: Route progress: 34/698 pending done (5% overall)
2026-03-25 16:55:52,335 [INFO] sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing.routing.route_builder: Route progress: 68/698 pending done (10% overall)
2026-03-25 16:55:55,952 [INFO] sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing.routing.route_builder: Route progress: 102/698 pending done (15% overall)
2026-03-25 16:55:59,537 [INFO] sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing.routing.route_builder: Route progress: 136/698 pending done (19% overall)
2026-03-25 16:56:03,904 [INFO] sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing.routing.route_builder: Route progress: 170/698 pending done (24% overall)
2026-03-25 16:56:

[TIMING] compute_routes: 69.19s
Starting route quality validation for: data/sum_nsm_dynamic_pricing/geneva_1km_mon08h_geometry/od_routes.json


2026-03-25 16:56:52,024 [INFO] sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing.quality: OD cleanup removed 166/698 OD pairs (23.78% invalid) from data/sum_nsm_dynamic_pricing/geneva_1km_mon08h_geometry/od_routes.json
2026-03-25 16:56:52,076 [INFO] sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing.quality: Route quality validation passed for data/sum_nsm_dynamic_pricing/geneva_1km_mon08h_geometry/od_routes.json (532 OD pairs)


Completed route quality validation for: data/sum_nsm_dynamic_pricing/geneva_1km_mon08h_geometry/od_routes.json - Validated: True


2026-03-25 16:56:52,715 [ERROR] sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing.quality: delta_ia for arc '484' is unbalanced (sum=1.0, expected 0)
2026-03-25 16:56:52,715 [ERROR] sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing.quality: delta_ia for arc '521' is unbalanced (sum=1.0, expected 0)
2026-03-25 16:56:52,716 [ERROR] sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing.quality: delta_ia for arc '618' is unbalanced (sum=1.0, expected 0)
2026-03-25 16:56:52,716 [ERROR] sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing.quality: delta_ia for arc '632' is unbalanced (sum=1.0, expected 0)
2026-03-25 16:56:52,716 [ERROR] sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing.quality: delta_ia for arc '640' is unbalanced (sum=1.0, expected 0)
2026-03-25 16:56:52,716 [ERROR] sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing.quality: delta_ia for arc '644' is unbalanced (sum=1.0, expected 0)
2026-03-25 16:56:52,717 [ERROR] sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing.qual

[TIMING] generate_instance: 1.45s
Instance generated at: data/sum_nsm_dynamic_pricing/geneva_1km_mon08h_geometry/Instance_532.json
[TIMING] total: 74.58s


## Optimization and Metrics Step
Run the exact optimization model on the generated instance and then export summary metrics.

In [2]:
from pathlib import Path
import time

from sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing import (
    InstancePipeline,
    OptimizerParams,
    PipelineConfig,
    run_pipeline,
    InstanceParams,
 )
from sum_nsm_dynamic_pricing.src.sum_nsm_dynamic_pricing.stations.providers import DonkeyRepublicProvider

instance_dir = Path("data/sum_nsm_dynamic_pricing/geneva_1km_mon08h_geometry")

config = PipelineConfig(
    center_lat=46.2059149,
    center_lon=6.1438511,
    radius_m=1000,
    output_dir=instance_dir,
    gtfs_path=Path("/Users/rebecamurillo/Documents/INRIA/development/inocs-sum-gtfs-geojson/otp/data/geneva-2026-gtfs.zip"),
    gbfs_path=None,
    seed=42,
    otp_timestamp="2026-03-08T08:30:00.000000Z",
 )

pipeline = InstancePipeline(config, gbfs_provider=DonkeyRepublicProvider())

# Optional: run the full instance generation pipeline if no instance exists yet.
instance_candidates = sorted(instance_dir.glob("Instance_*.json"))
if instance_candidates:
    instance_path = instance_candidates[-1]
else:
    instance_path = "NONE"

print(f"Using instance: {instance_path}")

params = OptimizerParams()

start = time.perf_counter()
solution_path = pipeline.optimize(
    params=params,
    instance_path=instance_path,
)

print(f"Optimization results processed in : {time.perf_counter() - start:.2f}s - {solution_path}")

start = time.perf_counter()
web_demo_path = pipeline.export_web_demo()
print(f"[TIMING] export_web_demo: {time.perf_counter() - start:.2f}s")
print(f"Web demo data exported to: {web_demo_path}")

metrics_path = pipeline.summarize_optimization(solution_path, params)
print(f"Optimization metrics: {metrics_path}")


Using instance: data/sum_nsm_dynamic_pricing/geneva_1km_mon08h_geometry/Instance_532.json
⚠️ Dropped 48 stations not used in delta_ia: ['bs_18394', 'bs_18416', 'bs_18496', 'bs_18497', 'bs_18509', 'bs_18543', 'bs_18574', 'bs_18589', 'bs_18593', 'bs_18603', 'bs_18611', 'bs_18613', 'bs_18807', 'bs_18835', 'bs_18837', 'bs_18840', 'bs_18843', 'bs_20712', 'bs_20714', 'bs_20715', 'bs_20720', 'bs_21480', 'bs_21485', 'bs_21869', 'bs_21905', 'bs_21907', 'bs_21909', 'bs_21911', 'bs_21936', 'bs_21938', 'bs_21960', 'bs_21965', 'bs_21967', 'bs_22185', 'bs_22186', 'bs_22187', 'bs_22188', 'bs_22207', 'bs_22210', 'bs_22237', 'bs_22313', 'bs_22314', 'bs_22363', 'bs_22364', 'bs_22365', 'bs_22366', 'bs_22741', 'bs_27876']
Set parameter Username


2026-03-25 16:59:01,433 [INFO] gurobipy: Set parameter Username


Set parameter LicenseID to value 2790753


2026-03-25 16:59:01,433 [INFO] gurobipy: Set parameter LicenseID to value 2790753


Academic license - for non-commercial use only - expires 2027-03-12


2026-03-25 16:59:01,434 [INFO] gurobipy: Academic license - for non-commercial use only - expires 2027-03-12


... (75 outputs left out)

2026-03-25 16:59:02,077 [INFO] gurobipy: Optimal solution found (tolerance 1.00e-04)


Best objective 2.446155395451e+03, best bound 2.446155395451e+03, gap 0.0000%


2026-03-25 16:59:02,078 [INFO] gurobipy: Best objective 2.446155395451e+03, best bound 2.446155395451e+03, gap 0.0000%


Saving solution files for 1 solution(s)...
✓ Saved: data/sum_nsm_dynamic_pricing/geneva_1km_mon08h_geometry/gurobi/solution_Instance_532.sol
✓ Saved: data/sum_nsm_dynamic_pricing/geneva_1km_mon08h_geometry/gurobi/solution_Instance_532.mst
✓ Saved: data/sum_nsm_dynamic_pricing/geneva_1km_mon08h_geometry/gurobi/solution_info_Instance_532.txt
Successfully saved 3 solution files
Solution time for Instance_532: 0.24 seconds
Solver ended Instance_532 with status 2. Best objective found: 2446.155395450711
Total feasible solutions found: 1
Solution saved to: data/sum_nsm_dynamic_pricing/geneva_1km_mon08h_geometry/Instance_532_0.6_None_invDefault_solution.json
Optimization results processed in : 0.97s - data/sum_nsm_dynamic_pricing/geneva_1km_mon08h_geometry/optimization_instance_results.json
[TIMING] export_web_demo: 7.14s
Web demo data exported to: data/sum_nsm_dynamic_pricing/geneva_1km_mon08h_geometry/web_demo
Optimization metrics: data/sum_nsm_dynamic_pricing/geneva_1km_mon08h_geometry/opt